# Comparing mixing matrices

The force of infection rebuilds a 121×121 kernel on every solver stage, projects it down to eight age bands, and divides by the spectral radius. This notebook compares that function, `get_norm_c_matrix`, with the three builders in `vacf.contact`. None of them is called by `run()`. The golden model is unchanged. The design notes are `plans/mixing-matrix.md` and `plans/mixing-spline.md`.

Use the **vacf (pixi)** kernel, as in [the parity notebook](01-InspectRun.ipynb).

**Baseline** truncates calendar time to an integer year and rebuilds the kernel.

**Stepwise** stores one demographic record per integer year. Once `bg_mixing`, `a_spread` and `pc_strength` are known it writes a normalised 8×8 table, and a stage indexes that table. The values match the baseline, including between 1 January dates, because the baseline is already constant on each year. The derivative in time is zero.

**Smooth** treats the annual weights, fertility and populations as point samples. A monotone cubic runs through them. The step evaluates 8×8 curves and only then divides by the spectral radius, so the radius stays 1 at fractional times. On 1 January the matrix matches the baseline. Between Januarys it moves, and that movement has a continuous time derivative.

**Spline** goes the other way round: it aggregates and normalises at each calendar year, and runs a Catmull-Rom cubic through the *normalised* matrices. The eigenproblem is then a per-draw cost, not a per-stage one, and a stage is a gather and a Horner cubic. It matches the baseline on 1 January like smooth, and it is C1 like smooth, but nothing is rebuilt inside the step. The spectral radius does not stay at 1 for free under that interpolation; the last section shows what it costs to put it back.

The plots below are the mixing matrix, not the epidemic.


## Build both from the Vietnam tables

The scalars are the inspect-notebook defaults. The arrays are the ones `load_static` closes over: within-band weights and group population for 1950–2101, fertility for 1950–2100.


In [1]:
from time import perf_counter

import jax
import numpy as np
import plotly.graph_objects as go
from jax import numpy as jnp
from plotly.subplots import make_subplots

from vacf.constants import AGE_STRATA
from vacf.contact import prepare_smooth, prepare_spline, prepare_stepwise
from vacf.mixing import canberra_distance, get_norm_c_matrix
from vacf.parameters import BASE_PARAMS
from vacf.static_data import load_static

static = load_static()
bg = float(BASE_PARAMS["bg_mixing"])
pc = float(BASE_PARAMS["pc_strength"])
a = float(BASE_PARAMS["a_spread"])
arrays = (
    static.age_weights,
    static.weight_ends,
    static.pops,
    static.pop_ends,
    static.fert,
    static.fert_ends,
)

stepwise = prepare_stepwise(*arrays)
smooth = prepare_smooth(*arrays)
spline = prepare_spline(*arrays).materialize(bg, a, pc)


def baseline_matrix(time: float) -> jax.Array:
    return get_norm_c_matrix(*arrays, time, bg, a, pc)


baseline_at = jax.jit(baseline_matrix)
step_at = jax.jit(lambda time: stepwise.matrix(time, bg, a, pc))
smooth_at = jax.jit(lambda time: smooth.matrix(time, bg, a, pc))
spline_at = jax.jit(spline.matrix)
for fn in (baseline_at, step_at, smooth_at, spline_at):
    jax.block_until_ready(fn(2000.0))

print(
    f"weight years {float(static.weight_ends[0]):.0f}–{float(static.weight_ends[1]):.0f}, "
    f"stepwise table years {stepwise.year0:.0f}–{stepwise.year1:.0f}, "
    f"smooth knots {smooth.knot_x.shape[0]}, spline knots {spline.n_knots}"
)


weight years 1950–2101, stepwise table years 1950–2101, smooth knots 605, spline knots 152


## Distance from the baseline

On a 0.1-year grid the stepwise matrix sits on the baseline: the truncation was already the model, and the year table copies it.

Both curved variants meet the baseline on each 1 January to about `1e-15`, and leave it in between, on purpose. They report the same maximum error, and that is not a coincidence: the largest gap is at the end of a year, where the baseline is still holding January's matrix while a continuous curve has almost reached next January's. What both numbers measure is the size of the baseline's own annual step, not a difference between the two variants.

They do differ, just not at the maximum. Smooth interpolates the demographic inputs and then aggregates; spline aggregates the measured years and then interpolates. Between years the two sit about `4e-4` apart on entries of size `1e-1`. Neither is the reference for the other — the underlying series are annual, and nothing in the data says what happens in July.

The entry plotted is the age-3 row and the age-40 column, one cell of the 8×8. The lower panel is the maximum absolute entry error against the baseline.


In [2]:
times = np.arange(1950.0, 2101.0 + 1e-9, 0.1)
row, col = 1, 6
entry_name = f"age {AGE_STRATA[row]} to age {AGE_STRATA[col]}"

baseline_entry = []
step_entry = []
smooth_entry = []
spline_entry = []
step_abs = []
smooth_abs = []
spline_abs = []
step_canberra = []
smooth_canberra = []
spline_canberra = []
for time in times:
    base = np.asarray(baseline_at(float(time)))
    step = np.asarray(step_at(float(time)))
    curved = np.asarray(smooth_at(float(time)))
    fitted = np.asarray(spline_at(float(time)))
    baseline_entry.append(base[row, col])
    step_entry.append(step[row, col])
    smooth_entry.append(curved[row, col])
    spline_entry.append(fitted[row, col])
    step_abs.append(float(np.max(np.abs(step - base))))
    smooth_abs.append(float(np.max(np.abs(curved - base))))
    spline_abs.append(float(np.max(np.abs(fitted - base))))
    step_canberra.append(float(canberra_distance(step, base)))
    smooth_canberra.append(float(canberra_distance(curved, base)))
    spline_canberra.append(float(canberra_distance(fitted, base)))

january = np.isclose(times % 1.0, 0.0, atol=1e-8)
print(
    f"stepwise max abs {max(step_abs):.3e}, max Canberra {max(step_canberra):.3e}"
)
print(
    f"smooth max abs {max(smooth_abs):.3e}, max Canberra {max(smooth_canberra):.3e}"
)
print(
    f"spline max abs {max(spline_abs):.3e}, max Canberra {max(spline_canberra):.3e}"
)
for name, values in (("smooth", smooth_abs), ("spline", spline_abs)):
    values = np.asarray(values)
    print(
        f"{name} max abs on 1 January {max(values[january]):.3e}, "
        f"between Januarys {max(values[~january]):.3e}"
    )

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        f"Normalised contact, {entry_name}",
        "Max absolute error against the baseline",
    ),
)
for name, values in (
    ("baseline", baseline_entry),
    ("stepwise", step_entry),
    ("smooth", smooth_entry),
    ("spline", spline_entry),
):
    fig.add_trace(
        go.Scatter(x=times, y=values, name=name, mode="lines"),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(x=times, y=step_abs, name="stepwise error", mode="lines"),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=times, y=smooth_abs, name="smooth error", mode="lines"),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=times, y=spline_abs, name="spline error", mode="lines"),
    row=2,
    col=1,
)
fig.update_xaxes(title_text="Calendar year", row=2, col=1)
fig.update_yaxes(title_text="Matrix entry", row=1, col=1)
fig.update_yaxes(title_text="Max |error|", row=2, col=1)
fig.update_layout(height=640, legend_traceorder="normal")
fig.show()


stepwise max abs 8.882e-16, max Canberra 4.977e-14
smooth max abs 1.658e-02, max Canberra 1.090e+00
spline max abs 1.658e-02, max Canberra 1.090e+00
smooth max abs on 1 January 2.082e-16, between Januarys 1.658e-02
spline max abs on 1 January 3.331e-16, between Januarys 1.658e-02


## Cost of a draw, and cost of a stage

Building the demographic records is once per process. Folding the three scalars is once per draw. A solver stage only evaluates.

`prepare` is host NumPy and is timed directly. The fold is timed under `jit`, because that is how a calibration calls it: eagerly it dispatches thirty-odd small kernels and the Python round trips swamp the arithmetic. A contact whose scalars were all fixed at `prepare` already holds its table or curve, so its fold is a return — the work happened inside `prepare`.

The evaluation column is two thousand separate dispatched calls. That number is floored by Python at a few microseconds whatever it runs, so it cannot separate the three builders; the cell after the figure times the same functions inside one `fori_loop`, which is what a stage actually costs.

Three parameterisations per builder: all three scalars fixed when the records are built; `bg_mixing` and `pc_strength` fixed with `a_spread` folded at the draw; all three folded at the draw. When `a_spread` is folded, the lag tensor is contracted and the stage does not read it.


In [3]:
eval_times = np.linspace(1950.0, 2101.0, 2000)


def token(contact: object) -> jax.Array:
    """Any array on the folded object, to block on."""
    curve = getattr(contact, "curve", None)
    if curve is not None:
        return curve.value
    table = getattr(contact, "table", None)
    if table is not None:
        return table
    offset = getattr(contact, "offset_y", None)
    if offset is None:
        offset = contact.offset
    return offset


def seconds(fn: object) -> tuple[float, object]:
    """Host time of one call. Used for `prepare`, which is NumPy."""
    ready = fn()
    jax.block_until_ready(token(ready))
    start = perf_counter()
    ready = fn()
    jax.block_until_ready(token(ready))
    return perf_counter() - start, ready


def fold_seconds(fn: object) -> tuple[float, object]:
    """Time the fold under jit, which is how a calibration calls it.

    Eagerly, `materialize` dispatches thirty-odd small kernels and the Python
    round trips dominate — for the spline that reads as twenty milliseconds
    for half a millisecond of arithmetic.
    """
    ready = fn()
    compiled = jax.jit(lambda: token(fn()))
    jax.block_until_ready(compiled())
    start = perf_counter()
    for _ in range(10):
        jax.block_until_ready(compiled())
    return (perf_counter() - start) / 10, ready


def microseconds(fn: object) -> float:
    jax.block_until_ready(fn(float(eval_times[0])))
    start = perf_counter()
    for time in eval_times:
        jax.block_until_ready(fn(float(time)))
    return (perf_counter() - start) / len(eval_times) * 1e6


cases = (
    ("stepwise, fixed", prepare_stepwise, dict(bg_mixing=bg, a_spread=a, pc_strength=pc), None),
    ("stepwise, a free", prepare_stepwise, dict(bg_mixing=bg, pc_strength=pc), lambda c: c.materialize(a_spread=a)),
    ("stepwise, all free", prepare_stepwise, {}, lambda c: c.materialize(bg, a, pc)),
    ("smooth, fixed", prepare_smooth, dict(bg_mixing=bg, a_spread=a, pc_strength=pc), None),
    ("smooth, a free", prepare_smooth, dict(bg_mixing=bg, pc_strength=pc), lambda c: c.materialize(a_spread=a)),
    ("smooth, all free", prepare_smooth, {}, lambda c: c.materialize(bg, a, pc)),
    ("spline, fixed", prepare_spline, dict(bg_mixing=bg, a_spread=a, pc_strength=pc), None),
    ("spline, a free", prepare_spline, dict(bg_mixing=bg, pc_strength=pc), lambda c: c.materialize(a_spread=a)),
    ("spline, all free", prepare_spline, {}, lambda c: c.materialize(bg, a, pc)),
)

labels = []
prepare_ms = []
materialize_ms = []
eval_us = []
for label, builder, fixed, fold in cases:
    prepare_s, contact = seconds(lambda: builder(*arrays, **fixed))
    if fold is None:
        materialize_s, ready = fold_seconds(lambda: contact.materialize())
    else:
        materialize_s, ready = fold_seconds(lambda: fold(contact))
    evaluate = jax.jit(lambda time: ready.matrix(time))
    labels.append(label)
    prepare_ms.append(prepare_s * 1e3)
    materialize_ms.append(materialize_s * 1e3)
    eval_us.append(microseconds(evaluate))
    print(
        f"{label:22}  prepare {prepare_ms[-1]:8.1f} ms   "
        f"materialize {materialize_ms[-1]:8.1f} ms   eval {eval_us[-1]:7.1f} µs"
    )

# The baseline has no separate draw. Time its stage the same way.
baseline_us = microseconds(baseline_at)
print(f"{'baseline stage':22}  eval {baseline_us:7.1f} µs")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Folding the scalars", "One solver stage"),
)
fig.add_trace(go.Bar(x=labels, y=materialize_ms, name="materialize"), row=1, col=1)
fig.add_trace(
    go.Bar(x=labels + ["baseline"], y=eval_us + [baseline_us], name="evaluation"),
    row=1,
    col=2,
)
fig.update_yaxes(title_text="Time (ms)", row=1, col=1)
fig.update_yaxes(title_text="Time (µs)", row=1, col=2)
fig.update_layout(height=420, showlegend=False)
fig.show()


stepwise, fixed         prepare     65.1 ms   materialize      0.0 ms   eval     9.3 µs


stepwise, a free        prepare     56.5 ms   materialize      1.1 ms   eval     9.1 µs


stepwise, all free      prepare     56.6 ms   materialize      1.1 ms   eval    11.7 µs


smooth, fixed           prepare    378.4 ms   materialize      0.0 ms   eval    47.5 µs


smooth, a free          prepare    392.3 ms   materialize      0.0 ms   eval    45.2 µs


smooth, all free        prepare    400.7 ms   materialize      0.0 ms   eval    43.6 µs


spline, fixed           prepare    106.4 ms   materialize      0.0 ms   eval    22.5 µs


spline, a free          prepare     54.4 ms   materialize      0.4 ms   eval    33.2 µs


spline, all free        prepare     71.9 ms   materialize      0.4 ms   eval    39.4 µs


baseline stage          eval   203.8 µs


### The same stages without Python dispatch

A dispatched `jax.jit` call costs a few microseconds whatever it runs, which is enough to hide every difference between the three builders. Summing each matrix over a `fori_loop` inside one jit pays that once and leaves the arithmetic.

Each builder is folded first, so this is the stage a calibration would pay: a gather for stepwise, a curve evaluation and an eigenproblem for smooth, a gather and a cubic for spline.

Read the result carefully, because it is not the argument for the spline. The stepwise gather is the cheapest thing here — cheaper than the spline's cubic, which has four gathers and a Rayleigh quotient in it. What the figure shows is that all three folded stages are somewhere between a thousandth and a tenth of one Dopri5 step, which is around 50 µs for this model. Once the matrix is precomputed, its arithmetic has stopped mattering.

The spline is not faster than the stepwise table because its stage is cheaper. It is faster because its stage is *continuous*, and the solver stops throwing steps away. [The runtime notebook](04-MixingRuntime.ipynb) is where that shows up.


In [4]:
from jax import lax

# Two thousand dispatched calls measure Python, not the stage. Sum the same
# function over a `fori_loop` inside one jit and the dispatch is paid once.
STAGE_STEPS = 20_000


def stage_us(fn: object) -> float:
    @jax.jit
    def total() -> jax.Array:
        def body(index: int, carried: jax.Array) -> jax.Array:
            return carried + fn(1950.0 + 151.0 * index / STAGE_STEPS)

        return lax.fori_loop(0, STAGE_STEPS, body, jnp.zeros((len(AGE_STRATA),) * 2))

    jax.block_until_ready(total())
    start = perf_counter()
    for _ in range(5):
        jax.block_until_ready(total())
    return (perf_counter() - start) / 5 / STAGE_STEPS * 1e6


# Fold each builder first. `stepwise.matrix(time, bg, a, pc)` with free
# scalars rebuilds one year from the lag tensor; that is not the stage a
# calibration pays, and it is not what the table row is about.
folded_stepwise = stepwise.materialize(bg, a, pc)
folded_smooth = smooth.materialize(bg, a, pc)

stage_names = ["baseline", "stepwise", "smooth", "spline"]
stage_costs = [
    stage_us(baseline_matrix),
    stage_us(folded_stepwise.matrix),
    stage_us(folded_smooth.matrix),
    stage_us(spline.matrix),
]
costs = dict(zip(stage_names, stage_costs, strict=True))
for name, cost in costs.items():
    print(f"{name:10} {cost:9.4f} µs per stage")
print(f"\nagainst the baseline stage: spline {costs['baseline'] / costs['spline']:.0f}x cheaper, "
      f"stepwise {costs['baseline'] / costs['stepwise']:.0f}x")
print(f"the stepwise gather is {costs['spline'] / costs['stepwise']:.1f}x cheaper than the spline cubic")
print("one Dopri5 step of this model is around 50 µs, so all three folded stages are noise")

fig = go.Figure(go.Bar(x=stage_names, y=stage_costs))
fig.update_layout(
    title="One stage, dispatch amortised",
    yaxis_title="Time (µs)",
    yaxis_type="log",
    height=380,
)
fig.show()

baseline     58.3759 µs per stage
stepwise      0.0848 µs per stage
smooth        8.6348 µs per stage
spline        0.2010 µs per stage

against the baseline stage: spline 290x cheaper, stepwise 689x
the stepwise gather is 2.4x cheaper than the spline cubic
one Dopri5 step of this model is around 50 µs, so all three folded stages are noise


## Time derivative

Between integers the stepwise derivative is zero: the index does not depend on the fractional year. The plotted entry itself jumps at 2000, and that jump is the baseline's. Both curved variants remove it, and their left- and right-hand derivatives agree at the knot.

That is not a cosmetic property. A Dopri5 step that straddles 1 January is evaluated over six stages against a field that changed underneath it, the error estimate rejects it, and the controller shrinks and tries again. At `rtol = 1e-9` the baseline rejects 4 195 of 8 986 steps. [The runtime notebook](04-MixingRuntime.ipynb) measures what removing the jump is worth.

The timing under the figure is `jax.grad` of one matrix entry against a forward evaluation, both after warmup. The stepwise reverse pass does not enter the eigenproblem, because the time cotangent is zero; it is cheaper than the forward pass. The smooth reverse pass does enter it. The spline reverse pass has no eigenproblem to enter: normalisation happened once, at the draw.


In [5]:
window = np.linspace(1999.0, 2001.0, 81)
grad_step = jax.jit(jax.grad(lambda time: stepwise.matrix(time, bg, a, pc)[row, col]))
grad_smooth = jax.jit(jax.grad(lambda time: smooth.matrix(time, bg, a, pc)[row, col]))
grad_spline = jax.jit(jax.grad(lambda time: spline.matrix(time)[row, col]))
for fn in (grad_step, grad_smooth, grad_spline):
    jax.block_until_ready(fn(2000.2))

step_slope = np.array([float(grad_step(float(time))) for time in window])
smooth_slope = np.array([float(grad_smooth(float(time))) for time in window])
spline_slope = np.array([float(grad_spline(float(time))) for time in window])

eps = 1e-4
knot = 2000.0
smooth_jump = float(grad_smooth(knot + eps) - grad_smooth(knot - eps))
spline_jump = float(grad_spline(knot + eps) - grad_spline(knot - eps))
step_value_jump = float(step_at(knot)[row, col] - step_at(knot - 1e-6)[row, col])
print(f"smooth derivative jump at {knot:.0f}: {smooth_jump:.3e}")
print(f"spline derivative jump at {knot:.0f}: {spline_jump:.3e}")
print(f"stepwise value jump at {knot:.0f}: {step_value_jump:.3e}")
print(f"max |stepwise slope| in the window: {np.max(np.abs(step_slope)):.3e}")
print(f"max |smooth slope| in the window: {np.max(np.abs(smooth_slope)):.3e}")
print(f"max |spline slope| in the window: {np.max(np.abs(spline_slope)):.3e}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=window, y=step_slope, name="stepwise d(entry)/dt", mode="lines"))
fig.add_trace(go.Scatter(x=window, y=smooth_slope, name="smooth d(entry)/dt", mode="lines"))
fig.add_trace(go.Scatter(x=window, y=spline_slope, name="spline d(entry)/dt", mode="lines"))
fig.update_layout(
    title=f"Time derivative of {entry_name}",
    xaxis_title="Calendar year",
    yaxis_title="d(entry)/d(year)",
    height=420,
)
fig.show()

forward_us = microseconds(jax.jit(lambda time: smooth.matrix(time, bg, a, pc)[row, col]))
reverse_us = microseconds(grad_smooth)
step_forward_us = microseconds(jax.jit(lambda time: stepwise.matrix(time, bg, a, pc)[row, col]))
step_reverse_us = microseconds(grad_step)
print(f"smooth forward {forward_us:.1f} µs, grad {reverse_us:.1f} µs, ratio {reverse_us / forward_us:.2f}")
print(
    f"stepwise forward {step_forward_us:.1f} µs, grad {step_reverse_us:.1f} µs, "
    f"ratio {step_reverse_us / step_forward_us:.2f}"
)
spline_forward_us = microseconds(jax.jit(lambda time: spline.matrix(time)[row, col]))
spline_reverse_us = microseconds(grad_spline)
print(
    f"spline forward {spline_forward_us:.1f} µs, grad {spline_reverse_us:.1f} µs, "
    f"ratio {spline_reverse_us / spline_forward_us:.2f}"
)


smooth derivative jump at 2000: 2.145e-07
spline derivative jump at 2000: 1.999e-07
stepwise value jump at 2000: 8.279e-04
max |stepwise slope| in the window: 0.000e+00
max |smooth slope| in the window: 1.250e-03
max |spline slope| in the window: 1.281e-03


smooth forward 58.7 µs, grad 77.9 µs, ratio 1.33
stepwise forward 43.2 µs, grad 6.9 µs, ratio 0.16


spline forward 23.6 µs, grad 39.2 µs, ratio 1.66


## What interpolating a normalised matrix costs, and how to get it back

`plans/mixing-matrix.md` ruled this out: a blend of two matrices whose spectral radius is 1 does not have spectral radius 1, and that within-year factor multiplies the force of infection and therefore the gradient of every infection parameter, not only the mixing scalars. The argument is right. What it lacked was a number.

`pin_radius=False` gives the drift. `pin_radius=True`, the default, also stores the dominant left and right vectors as curves and divides by the Rayleigh quotient `u C v / u v` of the interpolated matrix. That quotient is stationary at the true Perron pair, so an interpolated pair that is wrong by `e` gives a radius wrong by `e²` — which is why two matrix-vector products buy so much.


In [6]:
loose = prepare_spline(*arrays, pin_radius=False).materialize(bg, a, pc)
loose_at = jax.jit(loose.matrix)
for fn in (loose_at,):
    jax.block_until_ready(fn(2000.0))

drift_times = np.arange(float(static.weight_ends[0]), float(static.weight_ends[1]), 0.13)
pinned_drift = []
loose_drift = []
for when in drift_times:
    for values, fn in ((pinned_drift, spline_at), (loose_drift, loose_at)):
        radius = np.max(np.abs(np.linalg.eigvals(np.asarray(fn(float(when))))))
        values.append(abs(radius - 1.0))

print(f"over the whole table, max |rho - 1|: no pin {max(loose_drift):.3e}, pinned {max(pinned_drift):.3e}")
print(f"pinning is {max(loose_drift) / max(pinned_drift):.0f}x tighter")
moved = max(
    float(np.max(np.abs(spline_at(float(when)) - loose_at(float(when))))) for when in drift_times
)
print(f"it moves the matrix by at most {moved:.3e}, against an interpolation error of about 4e-4")
print(f"stage cost: pinned {stage_us(spline.matrix):.4f} µs, no pin {stage_us(loose.matrix):.4f} µs")

fig = go.Figure()
fig.add_trace(go.Scatter(x=drift_times, y=loose_drift, name="interpolated only", mode="lines"))
fig.add_trace(go.Scatter(x=drift_times, y=pinned_drift, name="Rayleigh-pinned", mode="lines"))
fig.update_layout(
    title="Departure of the spectral radius from one, between measurement years",
    xaxis_title="Calendar year",
    yaxis_title="|rho - 1|",
    yaxis_type="log",
    height=400,
)
fig.show()

over the whole table, max |rho - 1|: no pin 3.271e-05, pinned 1.473e-09
pinning is 22212x tighter
it moves the matrix by at most 1.608e-05, against an interpolation error of about 4e-4


stage cost: pinned 0.2090 µs, no pin 0.0642 µs


## No eigensolver anywhere

`normalise_by_spectral_radius` calls `jnp.linalg.eigvals`. That is a general complex eigensolver, applied to a matrix whose dominant root is real and simple by Perron–Frobenius; it is the most expensive single operation in the baseline stage; and JAX only lowers `eig` on CPU, so the present vector field cannot run on a GPU at all.

`vacf.contact.perron` replaces it with normalise-and-square. Squaring a matrix squares `|λ₂/λ₁|`, so `n` squarings are worth `2**n` power iterations, and ten of them are ten batched 8×8 products. The squared matrix is numerically rank one, so its row and column sums are the dominant vectors.

The cell below checks it where it is hardest. The corner of `PARAM_BOUNDS` with the smallest `bg_mixing`, `a_spread` and `pc_strength` is a nearly diagonal matrix, and its two leading roots are within 10% of each other. Plain power iteration is still wrong by `1e-5` there after 32 iterations; squaring reaches machine precision at every corner.


In [7]:
from itertools import product

from vacf.contact.perron import spectral_radius
from vacf.parameters import PARAM_BOUNDS

free_spline = prepare_spline(*arrays)
corners = list(product(*(PARAM_BOUNDS[name] for name in ("bg_mixing", "a_spread", "pc_strength"))))

rows = []
for corner_bg, corner_a, corner_pc in corners:
    raw = free_spline.knot_matrices(corner_bg, corner_a, corner_pc)
    reference = np.max(np.abs(np.linalg.eigvals(np.asarray(raw))), axis=-1)
    squared = np.asarray(spectral_radius(raw))
    eigenvalues = np.sort(np.abs(np.linalg.eigvals(np.asarray(raw))), axis=-1)[:, ::-1]
    rows.append(
        (
            corner_bg,
            corner_a,
            corner_pc,
            float(np.max(eigenvalues[:, 1] / eigenvalues[:, 0])),
            float(np.max(np.abs(squared - reference) / reference)),
        )
    )

print(f"{'bg':>7} {'a':>6} {'pc':>6}   worst |l2/l1|   max rel error")
for corner_bg, corner_a, corner_pc, gap, error in rows:
    print(f"{corner_bg:7.4f} {corner_a:6.2f} {corner_pc:6.3f}        {gap:8.4f}      {error:.2e}")

knots = free_spline.knot_matrices(bg, a, pc)
eig_call = jax.jit(lambda m: jnp.max(jnp.abs(jnp.linalg.eigvals(m)), axis=-1))
squaring_call = jax.jit(spectral_radius)
for fn in (eig_call, squaring_call):
    jax.block_until_ready(fn(knots))


def radius_ms(fn: object) -> float:
    start = perf_counter()
    for _ in range(20):
        jax.block_until_ready(fn(knots))
    return (perf_counter() - start) / 20 * 1e3


print(
    f"\n{knots.shape[0]} knots: eigvals {radius_ms(eig_call):.2f} ms, "
    f"squaring {radius_ms(squaring_call):.2f} ms"
)

     bg      a     pc   worst |l2/l1|   max rel error
 0.0010   2.00  0.050          0.9035      2.95e-15
 0.0010   2.00  1.000          0.7024      2.77e-15
 0.0010  15.00  0.050          0.5638      3.21e-15
 0.0010  15.00  1.000          0.4010      2.62e-15
 0.0200   2.00  0.050          0.6496      2.60e-15
 0.0200   2.00  1.000          0.5451      2.59e-15
 0.0200  15.00  0.050          0.2993      2.61e-15
 0.0200  15.00  1.000          0.2661      2.58e-15

152 knots: eigvals 1.87 ms, squaring 0.39 ms


### The gradient is the perturbation formula, not an approximation of it

The dominant vectors carry `stop_gradient`, so differentiating the Rayleigh quotient gives `u (dC) v / u v`. For a simple dominant root that *is* the derivative of the eigenvalue, so nothing is lost by not differentiating through the iteration — and a reverse pass through ten matrix squarings is avoided.

That is what makes `materialize` the right place for the fold: the whole per-draw path stays differentiable in all three scalars, and its reverse pass is cheaper than the stepwise table's, which has to push a cotangent back through `eig`.


In [8]:
def folded_entries(scalars: jax.Array) -> jax.Array:
    fitted = free_spline.materialize(scalars[0], scalars[1], scalars[2])
    return fitted.matrix(1980.3)[2, 5] + fitted.matrix(2040.0)[6, 1]


start_scalars = jnp.array([bg, a, pc])
analytic = np.asarray(jax.jit(jax.grad(folded_entries))(start_scalars))
step = 1e-6
numerical = np.array(
    [
        (
            float(folded_entries(start_scalars.at[index].add(step)))
            - float(folded_entries(start_scalars.at[index].add(-step)))
        )
        / (2.0 * step)
        for index in range(3)
    ]
)
print(f"{'':12}{'bg_mixing':>14}{'a_spread':>14}{'pc_strength':>14}")
print(f"{'jax.grad':12}" + "".join(f"{value:14.8f}" for value in analytic))
print(f"{'central diff':12}" + "".join(f"{value:14.8f}" for value in numerical))
print(f"{'rel error':12}" + "".join(f"{value:14.2e}" for value in np.abs(analytic / numerical - 1.0)))

free_stepwise = prepare_stepwise(*arrays)
fold_spline = jax.jit(lambda s: free_spline.materialize(s[0], s[1], s[2]).curve.value)
fold_step = jax.jit(lambda s: free_stepwise.materialize(s[0], s[1], s[2]).table)
grad_fold_spline = jax.jit(jax.grad(lambda s: jnp.sum(fold_spline(s) ** 2)))
grad_fold_step = jax.jit(jax.grad(lambda s: jnp.sum(fold_step(s) ** 2)))


def fold_ms(fn: object) -> float:
    jax.block_until_ready(fn(start_scalars))
    start = perf_counter()
    for _ in range(30):
        jax.block_until_ready(fn(start_scalars))
    return (perf_counter() - start) / 30 * 1e3


print(
    f"\nper draw: spline fold {fold_ms(fold_spline):.2f} ms, "
    f"reverse {fold_ms(grad_fold_spline):.2f} ms"
)
print(
    f"per draw: stepwise fold {fold_ms(fold_step):.2f} ms, "
    f"reverse {fold_ms(grad_fold_step):.2f} ms  (the eig VJP)"
)

                 bg_mixing      a_spread   pc_strength
jax.grad        1.86434695    0.01240299    0.07515267
central diff    1.86434676    0.01240298    0.07515260
rel error         1.05e-07      7.76e-07      9.93e-07



per draw: spline fold 1.17 ms, reverse 0.85 ms


per draw: stepwise fold 1.38 ms, reverse 2.43 ms  (the eig VJP)
